In [ ]:
# Submission path setup: run notebooks from any submission subfolder.
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'Functions.ipynb').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("default")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)


In [ ]:
csv_path = Path("early_search/evaluation_wide.csv")
assert csv_path.exists(), f"Could not find {csv_path.resolve()}"

df_raw = pd.read_csv(csv_path)
allowed_batch_sizes = [1, 2, 5]
df = df_raw[df_raw["batch_size"].isin(allowed_batch_sizes)].copy()
excluded_df = df_raw[~df_raw["batch_size"].isin(allowed_batch_sizes)].copy()

print("csv:", csv_path.resolve())
print("original shape:", df_raw.shape)
print("filtered shape:", df.shape)
print("excluded rows:", len(excluded_df))
print("excluded batch sizes:", sorted(excluded_df["batch_size"].dropna().unique().tolist()))
display(df.head())


In [ ]:
for col in ["arch_name", "loss_name", "batch_size", "lr", "weight_decay"]:
    print(f"{col}: {sorted(df[col].dropna().unique().tolist())}")


In [ ]:
df["dice_gap_train_val"] = df["dice_train"] - df["dice_val"]
df["dice_gap_val_test"] = df["dice_val"] - df["dice_test"]
df["loss_gap_val_train"] = df["loss_val"] - df["loss_train"]
df["hd95_gap_val_train"] = df["hd95_val"] - df["hd95_train"]

display(
    df[[
        "dice_train", "dice_val", "dice_test",
        "loss_train", "loss_val", "loss_test",
        "hd95_train", "hd95_val", "hd95_test",
        "dice_gap_train_val", "dice_gap_val_test",
        "loss_gap_val_train", "hd95_gap_val_train",
    ]].describe().round(4).T
)


In [ ]:
def summarize_param(param):
    summary = (
        df.groupby(param)
        .agg(
            runs=("run_name", "count"),
            loss_train_mean=("loss_train", "mean"),
            loss_val_mean=("loss_val", "mean"),
            loss_test_mean=("loss_test", "mean"),
            dice_train_mean=("dice_train", "mean"),
            dice_val_mean=("dice_val", "mean"),
            dice_test_mean=("dice_test", "mean"),
            hd95_train_mean=("hd95_train", "mean"),
            hd95_val_mean=("hd95_val", "mean"),
            hd95_test_mean=("hd95_test", "mean"),
        )
        .round(4)
    )
    return summary

for param in ["arch_name", "loss_name", "lr", "weight_decay", "batch_size"]:
    print(f"Summary by {param}")
    display(summarize_param(param))


In [ ]:
def ordered_values(param):
    vals = df[param].dropna().unique().tolist()
    if param in ["batch_size", "lr", "weight_decay"]:
        return sorted(vals)
    return sorted(vals, key=str)


pretty_param_names = {
    "arch_name": "Architecture",
    "loss_name": "Loss Function",
    "lr": "Learning Rate",
    "weight_decay": "Weight Decay",
    "batch_size": "Batch Size",
}

pretty_metric_names = {
    "Loss": "Loss (lower is better)",
    "Dice": "Dice (higher is better)",
    "HD95": "HD95 Distance (lower is better)",
}


def plot_grouped_boxplots_for_param(param):
    order = ordered_values(param)
    labels = [str(v) for v in order]
    param_label = pretty_param_names.get(param, param)

    metric_sets = [
        ("Loss", ["loss_train", "loss_val", "loss_test"]),
        ("Dice", ["dice_train", "dice_val", "dice_test"]),
        ("HD95", ["hd95_train", "hd95_val", "hd95_test"]),
    ]
    split_labels = ["Train", "Validation", "Test"]
    split_colors = ["#4C72B0", "#55A868", "#C44E52"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.6))

    for ax, (metric_name, cols) in zip(axes, metric_sets):
        width = 0.22
        centers = np.arange(len(order))
        offsets = [-width, 0.0, width]

        for offset, col, split_label, color in zip(offsets, cols, split_labels, split_colors):
            data = [df.loc[df[param] == val, col].dropna().values for val in order]
            positions = centers + offset
            bp = ax.boxplot(
                data,
                positions=positions,
                widths=0.2,
                patch_artist=True,
                showmeans=True,
                manage_ticks=False,
            )
            for patch in bp["boxes"]:
                patch.set_facecolor(color)
                patch.set_alpha(0.6)
            for median in bp["medians"]:
                median.set_color("black")
            for mean in bp["means"]:
                mean.set_marker("o")
                mean.set_markerfacecolor("black")
                mean.set_markeredgecolor("black")

        ax.set_xticks(centers)
        ax.set_xticklabels(labels)
        ax.set_xlabel(param_label)
        ax.set_ylabel(pretty_metric_names[metric_name])
        ax.set_title(f"{pretty_metric_names[metric_name]} across {param_label}")

    handles = [plt.Line2D([0], [0], color=c, lw=8, alpha=0.6) for c in split_colors]
    fig.suptitle(f"Train / Validation / Test distributions across {param_label}", y=0.98, fontsize=14)
    fig.legend(handles, split_labels, title="Data Split", loc="upper center", bbox_to_anchor=(0.5, 0.94), ncol=3, frameon=False)
    plt.tight_layout(rect=[0, 0, 1, 0.84])
    plt.show()


In [ ]:
for param in ["arch_name", "loss_name", "lr", "weight_decay", "batch_size"]:
    plot_grouped_boxplots_for_param(param)


In [ ]:
def scatter_by_arch(ax, x_col, y_col, title, x_label=None, y_label=None):
    for arch in sorted(df["arch_name"].unique()):
        sub = df[df["arch_name"] == arch]
        ax.scatter(sub[x_col], sub[y_col], label=arch, alpha=0.75)

    all_vals = np.concatenate([df[[x_col, y_col]].to_numpy().ravel()])
    lo = float(np.nanmin(all_vals))
    hi = float(np.nanmax(all_vals))
    pad = (hi - lo) * 0.05 if hi > lo else 1.0
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], linestyle="--", color="gray", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel(x_label or x_col)
    ax.set_ylabel(y_label or y_col)
    ax.legend()


fig, axes = plt.subplots(2, 2, figsize=(13, 10))
scatter_by_arch(axes[0, 0], "dice_val", "dice_test", "Validation Dice vs test Dice")
scatter_by_arch(axes[0, 1], "dice_train", "dice_val", "Train Dice vs validation Dice")
scatter_by_arch(axes[1, 0], "hd95_val", "hd95_test", "Validation HD95 vs test HD95")
scatter_by_arch(axes[1, 1], "hd95_train", "hd95_val", "Train HD95 vs validation HD95")
plt.tight_layout()
plt.show()


In [ ]:
split_info = [
    ("Train", "loss_train", "dice_train", "hd95_train", "#4C72B0"),
    ("Validation", "loss_val", "dice_val", "hd95_val", "#55A868"),
    ("Test", "loss_test", "dice_test", "hd95_test", "#C44E52"),
]


def corr_text(x, y):
    valid = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(valid) < 2:
        return "n/a"
    return f"r={valid['x'].corr(valid['y']):.3f}"


for loss_name in sorted(df["loss_name"].unique()):
    sub = df[df["loss_name"] == loss_name].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for split_label, loss_col, dice_col, hd95_col, color in split_info:
        axes[0].scatter(sub[loss_col], sub[dice_col], alpha=0.75, color=color, label=f"{split_label} ({corr_text(sub[loss_col], sub[dice_col])})")
        axes[1].scatter(sub[loss_col], sub[hd95_col], alpha=0.75, color=color, label=f"{split_label} ({corr_text(sub[loss_col], sub[hd95_col])})")

    axes[0].set_title(f"{loss_name}: Loss vs Dice")
    axes[0].set_xlabel("Loss")
    axes[0].set_ylabel("Dice")
    axes[0].legend(title="Data Split")

    axes[1].set_title(f"{loss_name}: Loss vs HD95")
    axes[1].set_xlabel("Loss")
    axes[1].set_ylabel("HD95 Distance")
    axes[1].legend(title="Data Split")

    fig.suptitle(f"Relationship between loss and evaluation metrics for {loss_name}", y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
arch_info = [
    ("shallow", "#4C72B0"),
    ("small", "#55A868"),
    ("wide", "#C44E52"),
]


for loss_name in sorted(df["loss_name"].unique()):
    sub = df[df["loss_name"] == loss_name].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for arch_name, color in arch_info:
        arch_sub = sub[sub["arch_name"] == arch_name]
        axes[0].scatter(arch_sub["loss_val"], arch_sub["dice_val"], alpha=0.75, color=color, label=f"{arch_name} ({corr_text(arch_sub['loss_val'], arch_sub['dice_val'])})")
        axes[1].scatter(arch_sub["loss_val"], arch_sub["hd95_val"], alpha=0.75, color=color, label=f"{arch_name} ({corr_text(arch_sub['loss_val'], arch_sub['hd95_val'])})")

    axes[0].set_title(f"{loss_name}: Validation loss vs validation Dice")
    axes[0].set_xlabel("Validation Loss")
    axes[0].set_ylabel("Validation Dice")
    axes[0].legend(title="Architecture")

    axes[1].set_title(f"{loss_name}: Validation loss vs validation HD95")
    axes[1].set_xlabel("Validation Loss")
    axes[1].set_ylabel("Validation HD95 Distance")
    axes[1].legend(title="Architecture")

    fig.suptitle(f"Architecture grouping within {loss_name}", y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
selection_df = df.sort_values(
    ["dice_val", "hd95_val", "loss_val"],
    ascending=[False, True, True],
).copy()

best_cols = [
    "run_name", "arch_name", "loss_name", "channels", "strides", "lr", "weight_decay", "batch_size",
    "loss_train", "loss_val", "loss_test",
    "dice_train", "dice_val", "dice_test",
    "hd95_train", "hd95_val", "hd95_test",
]

print("Top 15 overall by validation selection rule")
display(selection_df[best_cols].head(15))

print("Best per architecture")
best_per_arch = selection_df.groupby("arch_name", as_index=False).first()
display(best_per_arch[best_cols])

print("Best per architecture and loss")
best_per_arch_loss = selection_df.groupby(["arch_name", "loss_name"], as_index=False).first()
display(best_per_arch_loss[best_cols])
